# Fox timelines

Associated chart: the Caitlin Clark and Brittney Griner timelines, and the trans-coverage scrolly

Output files:
`fox_timeline.json`, `fox_nba_monthly.json`, and `trans_articles.csv` in `src/data/`.

Input files: `data/fox_wnba.csv`, `data/fox_nba.csv`, `data/topic_terms.csv`.


## Setup

In [ ]:
import csv, json, re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

DATA = Path("../data")
OUT = Path("../src/data")

csv.field_size_limit(10 ** 9)
fox = list(csv.DictReader((DATA / "fox_wnba.csv").open()))
len(fox)

## Fox WNBA timeline


In [ ]:
# the topic vocabulary is in data/topic_terms.csv 
from topics import patterns
TOPICS = list(patterns().items())

PLAYER_HL = [
    ("Caitlin Clark", r"caitlin clark|caitlin"),
    ("Sophie Cunningham", r"sophie cunningham|cunningham"),
    ("Brittney Griner", r"brittney griner|griner"),
]

# Fox splices ALL-CAPS links to other stories into the prose
SHOUT_RUN = re.compile(r"\b(?:[A-Z][A-Z'\u2019.&-]{1,}[,:;?!]?\s+){3,}[A-Z][A-Z'\u2019.&-]{1,}[,:;?!]?")


def trim(s, limit=220):
    s = re.sub(r"\s+", " ", s).strip()
    return s if len(s) <= limit else s[:limit].rsplit(" ", 1)[0] + "\u2026"


def proving_sentence(body, title, pattern):
    """The first readable sentence where this topic actually shows up.

    Returns (sentence, proved) — `proved` is False when we fell back to the
    headline, which lets the caller keep looking at other articles that month.
    """
    for candidate in re.split(r"(?<=[.!?])\s+", (body or "").replace("\n", " ")):
        candidate = re.sub(r"\s{2,}", " ", SHOUT_RUN.sub(" ", candidate)).strip(" \u2014-\u2013,;:")
        if not (40 <= len(candidate) <= 400):
            continue
        if re.search(pattern, candidate, re.I):
            return trim(candidate, 230), True
    return trim(title or "", 230), bool(re.search(pattern, title or "", re.I))

In [ ]:
months = defaultdict(int)
example = {}
example_plain = {}   # a headline naming no tracked player, so the WNBA
                     # example never duplicates a player example
player_rx = [re.compile(p, re.I) for _, p in PLAYER_HL]
mtopics = {t: defaultdict(int) for t, _ in PLAYER_HL}
mtopic_ex = {t: {} for t, _ in PLAYER_HL}   # player -> month -> proving sentence
mtopic_hed = {t: {} for t, _ in PLAYER_HL}  # player -> month -> a headline that itself matches
mtopic_rank = {t: {} for t, _ in PLAYER_HL}  # (headline matches, sentence proves)

for r in fox:
    d = (r.get("date") or "")[:7]
    # the study window starts in 2019; tag-feed stragglers from earlier
    # would stretch the timeline axis into near-empty years
    if not d or len(d) < 7 or d < "2019":
        continue
    months[d] += 1
    hed = r.get("headline", "")
    if hed and len(example.setdefault(d, [])) < 4:
        example[d].append(hed)
    if d not in example_plain and hed and not any(rx.search(hed) for rx in player_rx):
        example_plain[d] = hed
    text = ((r.get("body") or "") + " " + (r.get("headline") or "")).lower()
    for t, pat in PLAYER_HL:
        if re.search(pat, text):
            mtopics[t][d] += 1
            # best example: an article whose HEADLINE matches, with a proving
            # sentence; a headline that doesn't match is never shown as one
            best = mtopic_rank[t].get(d, (-1, -1))
            if best < (1, 1):
                hed = r.get("headline") or ""
                hed_match = 1 if re.search(pat, hed.lower()) else 0
                sentence, proved = proving_sentence(r.get("body"), hed, pat)
                rank = (hed_match, 1 if proved else 0)
                if rank > best:
                    mtopic_rank[t][d] = rank
                    mtopic_ex[t][d] = sentence
                    mtopic_hed[t][d] = trim(hed, 120) if hed_match else ""

# dense month axis from first to last
def next_month(m):
    y, mo = int(m[:4]), int(m[5:7])
    return f"{y + (mo == 12)}-{(mo % 12) + 1:02d}"

ms = sorted(months)
axis, cur = [], ms[0]
while cur <= ms[-1]:
    player_heds = {mtopic_hed[t].get(cur, "") for t, _ in PLAYER_HL} - {""}
    fallback = next((h for h in example.get(cur, []) if h not in player_heds),
                    (example.get(cur) or [""])[0])
    axis.append({"month": cur, "n": months.get(cur, 0),
                 "ex": example_plain.get(cur) or fallback,
                 "t": [mtopics[t].get(cur, 0) for t, _ in PLAYER_HL],
                 "tex": [mtopic_ex[t].get(cur, "") for t, _ in PLAYER_HL],
                 "thed": [mtopic_hed[t].get(cur, "") for t, _ in PLAYER_HL]})
    cur = next_month(cur)

#### Overwrites `src/data/fox_timeline.json`

In [ ]:
timeline_wnba = {"months": axis, "total": sum(months.values()),
                 "players": [t for t, _ in PLAYER_HL]}
print(f"fox_timeline: {len(axis)} months, {sum(months.values())} articles, {ms[0]}..{ms[-1]}")

pd.DataFrame([{"month": m["month"], "articles": m["n"],
               **{t: m["t"][i] for i, (t, _) in enumerate(PLAYER_HL)}}
              for m in axis]).tail(8)

## Fox NBA backdrop

Fox NBA articles per month from `data/fox_nba.csv`, aligned to the WNBA timeline's
month axis so the chart can join on index.

In [ ]:
nba = list(csv.DictReader((DATA / "fox_nba.csv").open()))
by_month = Counter(r["date"][:7] for r in nba if len(r.get("date") or "") >= 7)
nba_example = {}
for r in sorted(nba, key=lambda r: r.get("date") or ""):
    m = (r.get("date") or "")[:7]
    if len(m) == 7 and m not in nba_example and (r.get("headline") or "").strip():
        nba_example[m] = r["headline"].strip()
month_axis = [m["month"] for m in axis]

#### Overwrites `src/data/fox_nba_monthly.json`

In [ ]:
json.dump({"wnba": timeline_wnba,
           "nba": {"from": min(by_month), "months": month_axis,
                   "counts": [by_month.get(m, 0) for m in month_axis],
                   "ex": {m: nba_example[m] for m in month_axis if m in nba_example}}},
          open(OUT / "timeline.json", "w"))

on_axis = sum(by_month.get(m, 0) for m in month_axis)
print(f"{len(nba):,} NBA articles; earliest month {min(by_month)}; {on_axis:,} fall on the chart's axis")
pd.DataFrame({"month": month_axis, "nba": [by_month.get(m, 0) for m in month_axis]}).tail(8)

## Trans coverage 

One row per Fox WNBA article matching the timeline's "transgender / trans" pattern
in the headline or body, sorted by date.

In [ ]:
TRANS = re.compile(patterns()["transgender / trans"], re.I)

rows = []
for r in fox:
    date = (r.get("date") or "")[:10]
    if len(date) < 10:
        continue
    text = (r.get("headline") or "") + " " + (r.get("body") or "")
    if TRANS.search(text):
        rows.append({"date": date, "month": date[:7],
                     "title": (r.get("headline") or "")[:160],
                     "url": r.get("url", "")})
rows.sort(key=lambda a: a["date"])

#### Overwrites `src/data/trans_articles.csv`

In [ ]:
with (OUT / "trans_articles.csv").open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["date", "month", "title", "url"])
    w.writeheader()
    w.writerows(rows)

print(f"trans_articles: {len(rows)} articles, {rows[0]['date']}..{rows[-1]['date']}")
pd.DataFrame(rows)[["date", "title"]].tail(8)

## Spot check

In [ ]:
import random

for r in random.sample(fox, 5):
    text = f"{r['headline']} {r.get('body') or ''}"
    fired = [name for name, pat in TOPICS if re.search(pat, text, re.I)]
    print(f"   {(r.get('date') or '')[:10]} {r['headline'][:78]}")
    print(f"      → {', '.join(fired) or '(no topic)'}")